# 🏡 Min-Max Normalization Workshop
## Team Name/Number: Group 4
## Team Members: 
- Davisen Veerasamy
- Carlos Gutierrez
- Rangeetha 
- Nnamdi Ikengah
---

## ❗ Why We Normalize: The Problem with Raw Feature Scales

In housing data, features like `Price` and `Lot_Size` can have values in the hundreds of thousands, while others like `Num_Bedrooms` range from 1 to 5. This creates problems when we use algorithms that depend on numeric magnitudes.

---

### ⚠️ What Goes Wrong Without Normalization

---

### 1. 🧭 K-Nearest Neighbors (KNN)

KNN uses the **Euclidean distance** formula:

$$
d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2 + \cdots}
$$

**Example:**

- $ \text{Price}_1 = 650{,}000, \quad \text{Price}_2 = 250{,}000 $
- $ \text{Bedrooms}_1 = 3, \quad \text{Bedrooms}_2 = 2 $

Now compute squared differences:

$$
(\text{Price}_1 - \text{Price}_2)^2 = (650{,}000 - 250{,}000)^2 = (400{,}000)^2 = 1.6 \times 10^{11}
$$
$$
(\text{Bedrooms}_1 - \text{Bedrooms}_2)^2 = (3 - 2)^2 = 1
$$

➡️ **Price dominates the distance calculation**, making smaller features like `Bedrooms` irrelevant.

---

### 2. 📉 Linear Regression

Linear regression estimates:

$$
y = \beta_1 \cdot \text{Price} + \beta_2 \cdot \text{Bedrooms} + \beta_3 \cdot \text{Lot\_Size} + \epsilon
$$

If `Price` has very large values:
- Gradient updates for $ \beta_1 $ will be **much larger**
- Gradient updates for $ \beta_2 $ (Bedrooms) will be **very small**

➡️ The model overfits high-magnitude features like `Price`.

---

### 3. 🧠 Neural Networks

A single neuron computes:

$$
z = w_1 \cdot \text{Price} + w_2 \cdot \text{Bedrooms} + w_3 \cdot \text{Lot\_Size}
$$

If:

- $ \text{Price} = 650{,}000 $
- $ \text{Bedrooms} = 3 $
- $ \text{Lot\_Size} = 8{,}000 $

Then:

$$
z \approx w_1 \cdot 650{,}000 + w_2 \cdot 3 + w_3 \cdot 8{,}000
$$

➡️ Even with equal weights, `Price` contributes **most of the activation**, making it difficult for the network to learn from other features.

---

### ✅ Solution: Min-Max Normalization

We apply the transformation:

$$
x_{\text{normalized}} = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}
$$

This scales all features to a common range (typically $[0, 1]$).

| Feature      | Raw Value | Min     | Max     | Normalized Value |
|--------------|-----------|---------|---------|------------------|
| Price        | 650,000   | 250,000 | 800,000 | 0.72             |
| Bedrooms     | 3         | 1       | 5       | 0.50             |
| Lot_Size     | 8,000     | 3,000   | 10,000  | 0.714            |

➡️ Now, **each feature contributes fairly** to model training or distance comparisons.

---

## 📌 Use Case: Housing Data
We are normalizing features from a real estate dataset to prepare it for machine learning analysis.

In [1]:
# 🔢 Load and display dataset
import pandas as pd
df = pd.read_csv('./data/housing_data.csv')
df.head()

,House_ID,Price,Area_sqft,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size
0,H100000,574507,1462,3,3,2002,4878
1,H100001,479260,1727,2,2,1979,4943
2,H100002,597153,1403,5,2,1952,5595
3,H100003,728454,1646,5,2,1992,9305
4,H100004,464876,853,1,1,1956,7407


### 🔎 Step 1 — Implement Min-Max Normalization on the Housing Dataset

In [2]:
# ✍️ Implement Min-Max Normalization manually here (no sklearn/numpy)
# Normalize: Price, Area_sqft, Num_Bedrooms, Num_Bathrooms, Lot_Size

def min_max_normalize(values):
    # 1. find the smallest and largest value
    col_min = min(values)
    col_max = max(values)
    # 2. edge case: constant column (max == min) would divide by zero
    if col_max == col_min:
        return [0.0 for _ in values]
    # 3. apply the formula to each value and return the new list
    return [ (x - col_min) / (col_max - col_min) for x in values]

cols = ['Price', 'Area_sqft', 'Num_Bedrooms', 'Num_Bathrooms', 'Lot_Size']
df_norm = df.copy()
for col in cols:
    df_norm[col] = min_max_normalize(df[col].tolist())

df_norm.head()


,House_ID,Price,Area_sqft,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size
0,H100000,0.485226,0.315789,0.50,1.0,2002,0.320814
1,H100001,0.387827,0.394588,0.25,0.5,1979,0.326191
2,H100002,0.508384,0.298246,1.00,0.5,1952,0.380129
3,H100003,0.642651,0.370503,1.00,0.5,1992,0.687045
4,H100004,0.373119,0.134701,0.00,0.0,1956,0.530030


In [3]:
# Evidence for Talking Points #1: ranges and KNN distance before vs. after normalization
ranges = pd.DataFrame({
    'Raw range': [df[col].max() - df[col].min() for col in cols],
    'Normalized range': [df_norm[col].max() - df_norm[col].min() for col in cols],
}, index=cols)
print(ranges)

# Share of the squared Euclidean distance between house 0 and house 1 contributed by each feature
def distance_shares(data):
    squared_diffs = [(data.loc[0, col] - data.loc[1, col]) ** 2 for col in cols]
    total = sum(squared_diffs)
    return [100 * sq / total for sq in squared_diffs]

pd.DataFrame({
    'Raw share of distance (%)': distance_shares(df),
    'Normalized share of distance (%)': distance_shares(df_norm),
}, index=cols).round(4)

               Raw range  Normalized range
Price             977909               1.0
Area_sqft           3363               1.0
Num_Bedrooms           4               1.0
Num_Bathrooms          2               1.0
Lot_Size           12088               1.0


,Raw share of distance (%),Normalized share of distance (%)
Price,99.9992,2.8902
Area_sqft,0.0008,1.8918
Num_Bedrooms,0.0000,19.0418
Num_Bathrooms,0.0000,76.1673
Lot_Size,0.0000,0.0088


### 🔎 Talking Points #1 — Why Normalization Matters

- **KNN: without normalization, "nearest house" just means "nearest price."** KNN compares houses using Euclidean distance. Between the first two houses in our dataset, `Price` accounts for **99.999%** of the squared distance, and bedrooms, bathrooms and lot size contribute almost nothing. After normalization the distance is shared across features (Bathrooms 76%, Bedrooms 19%, Price 3%, Area 2%), so every feature counts.

- **Gradient-based learning: equal scales give faster, more stable training.** In gradient descent, each weight's update is multiplied by its feature's value. Raw `Price` (100,000–1,077,909) produces updates thousands of times larger than `Num_Bathrooms` (1–3), so a single learning rate is either too big for Price (training becomes unstable) or too small for Bathrooms (it barely learns). With every feature in [0, 1], one learning rate works for all weights.

- **Housing features use very different units, and normalization removes that bias.** Before normalization the ranges were Price ≈ 977,909, Lot_Size ≈ 12,088, Area_sqft ≈ 3,363, Bedrooms = 4 and Bathrooms = 2. After normalization every feature has a range of exactly 1, so a feature's importance no longer depends on its units.


## 🧩 Challenge Extension: After Normalization, Which Features Matter Most?

You’ve normalized the housing features so they share a common scale.  
Now comes a common next step in ML workflows:

> **How do we identify the most important directions (principal components) in the data—and how might these relate to a target variable like `Price`?**

This introduces **Principal Component Analysis (PCA)**.

---

## 📚 PCA Theory (Conceptual)

### What PCA *is*
PCA is an **unsupervised** dimensionality reduction technique that:
- Finds **new axes** (principal components) that are **linear combinations** of your original features.
- Orders these axes so that:
  - **PC1** captures the **most variance** in the feature space,
  - **PC2** captures the next most variance, and so on,
  - Each PC is **orthogonal** (uncorrelated) with the previous ones.

### What PCA is *not*
PCA does **not** directly find features that “impact the target variable” because it does not use the target in its optimization.

However, you *can*:
- Compute PCs from the feature matrix **X** (after normalization),
- Then measure how PCs relate to the target **y** (e.g., correlation with `Price`, or a simple regression on PCs),
- Interpret which original features contribute most to PCs that are most related to **y**.

---

## 🧠 The Math (high level)
Given a centered feature matrix \(X\) (often standardized/normalized first):

1. Compute covariance matrix:
$$
\Sigma = \frac{1}{n-1}X^\top X
$$

2. Find eigenvectors (principal directions) and eigenvalues (variance captured):
$$
\Sigma v_i = \lambda_i v_i
$$

- $ v_i $ are **principal component directions** (loadings)
- $ \lambda_i $ are the **variance explained** by each component

---

## ✅ Why Normalize Before PCA?
PCA is sensitive to scale. Without normalization/standardization:
- A large-scale feature (e.g., `Price`) can dominate variance
- PCs will reflect units rather than structure

---

## 🎯 Student Challenge
Using the **housing dataset**:

1. Apply PCA to the normalized feature matrix \(X\) (exclude ID columns and the target).
2. Determine how many components are needed to explain **≥ 90%** of the variance.
3. Identify which original features contribute most to:
   - **PC1** and **PC2**, and
   - the **PC most correlated with the target** (`Price`).
4. Write a short interpretation:
   - “What does PC1 represent in housing terms?”
   - “Do the PCs that explain the most variance also relate most strongly to `Price`?”



### 🔗 How to Integrate This With Your Step 1 Normalization

- If you created normalized columns (e.g., `Area_sqft_norm`), use those in `candidate_features`.
- If you normalized in-place (overwriting original columns), you can use the original names.
- PCA should **not** include:
  - `House_ID` (identifier)
  - non-numeric categorical columns (unless encoded appropriately)
- Decide intentionally whether to include `Year_Built`:
  - It’s numeric, but it may behave differently than size/price-related features.

---

### ✅ Deliverable for the Challenge
Add a Markdown cell answering:

1. How many PCs explain at least **90%** variance?
2. Which features contribute most to **PC1** and **PC2**?
3. Which PC is most correlated with `Price`?
4. In plain language: what do you think PC1 represents?


In [4]:
# --- PCA Scaffolding (after you have a normalized DataFrame) ---
# Assumptions:
# 1) You already loaded housing_data.csv into df
# 2) You created normalized columns OR a normalized DataFrame (e.g., df_norm)
#
# If you normalized in-place, you can build df_norm by selecting only the normalized feature columns.

import pandas as pd
import numpy as np

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ---- Step A: Choose target and feature columns ----
target_col = 'Price'
candidate_feature_cols = ['Area_sqft', 'Num_Bedrooms', 'Num_Bathrooms', 'Lot_Size']

X = df_norm[candidate_feature_cols]
y = df_norm[target_col]

# ---- Step B: Ensure scaling appropriate for PCA ----
# X is already min-max normalized (Step 1), and PCA centers the data itself,
# so no extra scaling is applied.

# ---- Step C: Fit PCA ----
pca = PCA()
X_pca = pca.fit_transform(X)

# ---- Step D: Variance explained ----
explained = pca.explained_variance_ratio_
cumulative = explained.cumsum()
print("Explained variance ratio:", explained)
print("Cumulative explained variance:", cumulative)
n_90 = np.argmax(cumulative >= 0.90) + 1
print(f"Number of PCs to explain 90% variance: {n_90}")

# ---- Step E: Loadings (feature contributions to PCs) ----
loadings = pd.DataFrame(pca.components_.T, index=candidate_feature_cols, columns=[f'PC{i+1}' for i in range(len(candidate_feature_cols))])
print("Loadings (feature contributions to PCs):")
print(loadings)

# ---- Step F: Relate PCs to the target (simple correlation) ----
for i in range(X_pca.shape[1]):
    pc = X_pca[:, i]
    corr = np.corrcoef(pc, y)[0, 1]
    print(f"Correlation between PC{i+1} and {target_col}: {corr:.4f}")

Explained variance ratio: [0.49531854 0.36004519 0.08026    0.06437626]
Cumulative explained variance: [0.49531854 0.85536374 0.93562374 1.        ]
Number of PCs to explain 90% variance: 3
Loadings (feature contributions to PCs):
                    PC1       PC2       PC3       PC4
Area_sqft      0.001548  0.017423 -0.123563  0.992183
Num_Bedrooms   0.053396  0.998253  0.020288 -0.015087
Num_Bathrooms  0.998563 -0.053326 -0.005191 -0.001268
Lot_Size       0.004326 -0.018523  0.992116  0.123874
Correlation between PC1 and Price: -0.0114
Correlation between PC2 and Price: -0.0054
Correlation between PC3 and Price: 0.0172
Correlation between PC4 and Price: -0.0152


### 🧪 PCA on Raw vs. Normalized Features

To show why scaling matters for PCA, we fit PCA on the same four features **before** normalization and compare the explained variance.

In [5]:
pca_raw = PCA().fit(df[candidate_feature_cols])

print("Raw PC1 loadings:")
print(pd.Series(pca_raw.components_[0], index=candidate_feature_cols).abs().sort_values(ascending=False).round(3))

pd.DataFrame({
    'Raw: Explained Variance': pca_raw.explained_variance_ratio_,
    'Normalized: Explained Variance': pca.explained_variance_ratio_,
}, index=[f'PC{i+1}' for i in range(len(candidate_feature_cols))]).round(3)

Raw PC1 loadings:
Lot_Size         1.000
Area_sqft        0.008
Num_Bedrooms     0.000
Num_Bathrooms    0.000
dtype: float64


,Raw: Explained Variance,Normalized: Explained Variance
PC1,0.941,0.495
PC2,0.059,0.360
PC3,0.000,0.080
PC4,0.000,0.064


### ✅ Deliverable for the Challenge

1. How many PCs explain at least **90%** variance?
  - 3.
2. Which features contribute most to **PC1** and **PC2**?
  - For PC1: Num_Bathrooms
  - For PC2: Num_Bedrooms
3. Which PC is most correlated with `Price`?
  - PC3 (r = 0.017). Even though it is the most correlated, the correlation is essentially zero, so there is no real relationship.
4. In plain language: what do you think PC1 represents?
  - PC1 is basically just the number of bathrooms. It is not a combined idea of overall house size.
  - The features didnt combine because they're uncorrelated: more area doesn't mean more bedrooms or a bigger lot. So PCA had nothing to merge.
  - Bathrooms came first after min-max, because its few values (0, 0.5, 1) give it the largest variance of the four features, and PCA ranks PCs by variance.
  - The high-variance PCs do not necessarily relate to Price. PC1 and PC2 explain the most variance, but PC3 has the highest correlation with Price, and all of the correlations are near zero. PCA finds where the features vary most, not what drives the target.


### 🔎 Talking Points #2 — PCA Insights

**Talking point #1: The scaling choice shapes PCA**

PCA ranks components by variance, so how we scale the data decides which features come first. After min-max normalization, `Num_Bathrooms` (variance ≈ 0.17) and `Num_Bedrooms` (≈ 0.12) had the most spread because they only take a few values, so they became PC1 and PC2. If we had used StandardScaler instead, every feature would have variance 1, each PC would explain only about 25%, and we would need all four PCs to reach 90%.

**Talking point #2: PCA can only find structure that's already there**

The housing features are almost completely uncorrelated (a bigger area doesn't mean more bedrooms or a bigger lot), so PCA had nothing to combine. Each PC ended up as a single feature, with one loading near 1.0 (0.999, 0.998, 0.992, 0.992) and the rest near 0. PCA is most useful when features overlap, which isn't the case in this dataset.

**Talking point #3: High variance doesn't mean high importance for Price**

High variance ≠ importance for Price. PC1 and PC2 explain 85.5% of the variance, but PC3 has the highest correlation with Price, and even that is only r = 0.017. PCA doesn't look at the target, so it shows where the features vary, not what affects price.

**Talking point #4: Year_Built**

Year_Built is numeric, but it behaves differently than the other features. It was left out because it measures the age of the house, not its size. Its raw values (1950–2022) look large, but they only span 72 years, so the numbers themselves say little about the house.

# 📤 How to Submit the Assignment

Follow these instructions carefully to ensure your team submits a complete and well-organized assignment.

## Step 1: Get the Starter Repository

1. Clone the GitHub repository provided by the instructor.
2. Keep the **original repository name**. Do **not** rename the repository.

## Step 2: Complete the Workshop

3. Create **one shared Jupyter Notebook** for the entire team to work on.
4. At the top of **`MinMax_Normalization_Workshop.ipynb`**, clearly include:
   - Team Number
   - Full Name of each team member
5. Complete **all coding tasks, analyses, and written answers** in the same notebook.
6. Make sure all notebook cells execute successfully before submission.

## Step 3: Update the Project Documentation

7. Replace the existing **`README.md`** file with a new version that:
   - Introduces your team
   - Explains the purpose of the workshop
   - Describes the work you completed
   - Summarizes your findings and conclusions
   - Can be used as the visual aid for your team's **5-minute presentation**

## Step 4: Review Project Dependencies

8. Review the project's Python dependencies and decide whether the **`requirements.txt`** file needs to be updated.
   - If you installed additional packages, regenerate or update the file.
   - If no new dependencies were added, no changes may be necessary.

## Step 5: Publish Your Work

9. Push the completed repository to your own GitHub account.
10. Verify that all files have been uploaded correctly and that the repository is publicly accessible if required by the instructor.

---

## ✅ Submission Checklist

Before submitting, confirm that the following items are complete:

- [ ] Repository name has not been changed
- [ ] `MinMax_Normalization_Workshop.ipynb` contains all required work
- [ ] Team number and team member names appear at the top of the notebook
- [ ] All questions have been answered
- [ ] All code runs successfully
- [ ] `README.md` has been replaced with a project summary
- [ ] README can support a 5-minute presentation
- [ ] `requirements.txt` has been reviewed and updated if necessary
- [ ] Repository has been pushed to your GitHub account
- [ ] Final work has been reviewed by all team members

> **Important:** Submit a repository that is clean, complete, and easy to understand. The notebook and README should clearly demonstrate your team's work, findings, and understanding of Min-Max Normalization.